# Knee Osteoarthritis Classification - 4-Class Experiment
## Grade 0 & Grade 1 merged into Grade 0-1
## VGG16, VGG19, DenseNet121, DenseNet201, Swin Tiny, ConvNeXt + Multi-Head Attention Fusion + Grad-CAM

In [ ]:
import numpy as np
import torch

print("numpy:", np.__version__)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

torch.from_numpy(np.zeros((1,), dtype=np.float32))
print("numpy bridge ok")

if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
    print("capability:", torch.cuda.get_device_capability(0))
    x = torch.randn(1024, 1024, device="cuda")
    print("matmul ok:", float((x @ x)[0, 0]))
else:
    raise RuntimeError("CUDA not available -- check accelerator and restart.")

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
from tqdm import tqdm
import pandas as pd
import time
import os
from copy import deepcopy
import cv2
from matplotlib.colors import LinearSegmentedColormap

try:
    import timm
except ImportError:
    !pip install timm -q
    import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

## 2. Hyperparameters and Data Paths

In [ ]:
BATCH_SIZE = 32
LEARNING_RATE = 0.0001
NUM_EPOCHS = 50
NUM_CLASSES = 4
IMG_SIZE = 224
NUM_WORKERS = 4

DATASET_NAME = 'Dataset-gans-filtered'
TRAIN_DIR = f'/kaggle/input/datasets/abdelrahman0211/dataset-gans-filtered/Train with Gans'
VAL_DIR = f'/kaggle/input/datasets/abdelrahman0211/dataset-gans-filtered/Filtered Val'
TEST_DIR = f'/kaggle/input/datasets/abdelrahman0211/dataset-gans-filtered/Filtered test'

print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Image Size: {IMG_SIZE}x{IMG_SIZE}")
print(f"\nData paths:")
print(f"  Train: {TRAIN_DIR}")
print(f"  Val: {VAL_DIR}")
print(f"  Test: {TEST_DIR}")

## 3. Data Transforms and Augmentation

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.RandomErasing(p=0.3),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

val_test_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

print("Transforms configured")

## 4. Load Datasets and Create Data Loaders

In [ ]:
def remap_labels_4class(dataset):
    new_targets = []
    for label in dataset.targets:
        if label <= 1:
            new_targets.append(0)
        elif label == 2:
            new_targets.append(1)
        elif label == 3:
            new_targets.append(2)
        elif label == 4:
            new_targets.append(3)
        else:
            new_targets.append(label)
    dataset.targets = new_targets
    dataset.samples = [(path, new_targets[idx]) for idx, (path, _) in enumerate(dataset.samples)]
    dataset.imgs = dataset.samples
    dataset.classes = ['Grade 0-1', 'Grade 2', 'Grade 3', 'Grade 4']
    dataset.class_to_idx = {'Grade 0-1': 0, 'Grade 2': 1, 'Grade 3': 2, 'Grade 4': 3}
    return dataset

print("\n" + "="*70)
print("LOADING DATASETS")
print("="*70)

train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(root=VAL_DIR, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(root=TEST_DIR, transform=val_test_transforms)

train_dataset = remap_labels_4class(train_dataset)
val_dataset = remap_labels_4class(val_dataset)
test_dataset = remap_labels_4class(test_dataset)


train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(f'Train samples: {len(train_dataset)}')
print(f'Validation samples: {len(val_dataset)}')
print(f'Test samples: {len(test_dataset)}')
print(f'Classes: {train_dataset.classes}')

## 5. Model Definitions
### VGG16, VGG19, DenseNet121, DenseNet201, Swin Tiny, ConvNeXt

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from collections import Counter


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()


class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_acc):
        if self.best_score is None:
            self.best_score = val_acc
        elif val_acc < self.best_score + self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = val_acc
            self.counter = 0


def get_class_weights(train_labels, num_classes=4):
    class_counts = Counter(train_labels)
    total_samples = len(train_labels)
    weights = torch.zeros(num_classes)
    for class_idx in range(num_classes):
        count = class_counts.get(class_idx, 1)
        weights[class_idx] = total_samples / (num_classes * count)
    return weights


def create_vgg16(num_classes=4, pretrained=True):
    model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1 if pretrained else None)
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Sequential(
        nn.Linear(num_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),

        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),

        nn.Linear(256, num_classes)
    )
    return model


def create_vgg19(num_classes=4, pretrained=True):
    model = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1 if pretrained else None)
    num_features = model.classifier[6].in_features
    model.classifier[6] = nn.Sequential(
        nn.Linear(num_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),

        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),

        nn.Linear(256, num_classes)
    )
    return model


def create_densenet121(num_classes=4, pretrained=True):
    model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
    num_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(num_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),

        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),

        nn.Linear(256, num_classes)
    )
    return model


def create_densenet201(num_classes=4, pretrained=True):
    model = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1 if pretrained else None)
    num_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Linear(num_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),

        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),

        nn.Linear(256, num_classes)
    )
    return model


def create_swin_tiny(num_classes=4, pretrained=True):
    return timm.create_model('swin_tiny_patch4_window7_224', pretrained=pretrained, num_classes=num_classes)


def create_convnext(num_classes=4, pretrained=True):
    if pretrained:
        model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
    else:
        model = models.convnext_tiny(weights=None)
    num_features = model.classifier[2].in_features
    model.classifier[2] = nn.Sequential(
        nn.Linear(num_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),

        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),

        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),

        nn.Linear(256, num_classes)
    )
    return model


def create_efficientnetv2(num_classes=4, pretrained=True):
    from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
    model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Linear(in_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


def create_vit_b16(num_classes=4, pretrained=True):
    from torchvision.models import vit_b_16, ViT_B_16_Weights
    model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1 if pretrained else None)
    in_features = model.heads.head.in_features
    model.heads.head = nn.Sequential(
        nn.Linear(in_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


def create_regnety008(num_classes=4, pretrained=True):
    from torchvision.models import regnet_y_800mf, RegNet_Y_800MF_Weights
    model = regnet_y_800mf(weights=RegNet_Y_800MF_Weights.IMAGENET1K_V1 if pretrained else None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Linear(in_features, 1024),
        nn.BatchNorm1d(1024),
        nn.ReLU(),
        nn.Dropout(0.6),
        nn.Linear(1024, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),
        nn.Dropout(0.5),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


MODEL_DICT = {
    'VGG16': create_vgg16,
    'VGG19': create_vgg19,
    'DenseNet121': create_densenet121,
    'DenseNet201': create_densenet201,
    'SwinTiny': create_swin_tiny,
    'ConvNeXt': create_convnext,
    'EfficientNetV2': create_efficientnetv2,
    'ViT_B16': create_vit_b16,
    'RegNetY008': create_regnety008,
}

print("Models loaded")
print(f"Available models: {list(MODEL_DICT.keys())}")

## 6. Training and Validation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, model_name=''):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(loader, desc=f'Training {model_name}')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, device, model_name=''):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc=f'Validating {model_name}'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100. * correct / total
    return epoch_loss, epoch_acc


def test_model(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc='Testing'):
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds)

print("Training and validation functions defined")

## 7. Helper Function for Training Individual Models

In [ ]:
print("=" * 60)
print("RUNNING 4-CLASS EXPERIMENT")
print("Classes: Grade 0-1 (merged) | Grade 2 | Grade 3 | Grade 4")
print("=" * 60)

results_summary = []

def train_single_model(model_name, model_fn, train_loader_to_use, val_loader_to_use, test_loader_to_use):
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print(f"{'='*70}")

    start_time = time.time()

    model = model_fn(num_classes=NUM_CLASSES, pretrained=True)
    model = model.to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters: {total_params:,}')
    print(f'Trainable parameters: {trainable_params:,}')


    if model_name == 'ConvNeXt':
        criterion = FocalLoss(gamma=2.0)
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    best_val_acc = 0.0
    train_history = {'loss': [], 'acc': []}
    val_history = {'loss': [], 'acc': []}

    early_stopping = EarlyStopping(patience=10)

    for epoch in range(NUM_EPOCHS):
        print(f'\nEpoch [{epoch+1}/{NUM_EPOCHS}]')

        train_loss, train_acc = train_epoch(model, train_loader_to_use, criterion, optimizer, device, model_name)
        val_loss, val_acc = validate(model, val_loader_to_use, criterion, device, model_name)

        train_history['loss'].append(train_loss)
        train_history['acc'].append(train_acc)
        val_history['loss'].append(val_loss)
        val_history['acc'].append(val_acc)

        print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')

        scheduler.step(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f'best_{model_name.lower().replace("/", "_")}_4class.pth')
            print(f'Best model saved! Val Acc: {best_val_acc:.2f}%')

        early_stopping(val_acc)
        if early_stopping.early_stop:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

    print(f'\nTesting {model_name}...')
    model.load_state_dict(torch.load(f'best_{model_name.lower().replace("/", "_")}_4class.pth', weights_only=True))
    test_labels, test_preds = test_model(model, test_loader_to_use, device)
    test_acc = accuracy_score(test_labels, test_preds) * 100

    training_time = time.time() - start_time

    class_names = ['Grade 0-1', 'Grade 2', 'Grade 3', 'Grade 4']
    print(f'\n{model_name} Classification Report:')
    print(classification_report(test_labels, test_preds, target_names=class_names))

    result = {
        'Model': model_name,
        'Parameters': total_params,
        'Best Val Acc': best_val_acc,
        'Test Acc': test_acc,
        'Training Time (min)': training_time / 60
    }
    results_summary.append(result)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(train_history['loss'], label='Train Loss', linewidth=2)
    axes[0].plot(val_history['loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'{model_name} - Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(train_history['acc'], label='Train Acc', linewidth=2)
    axes[1].plot(val_history['acc'], label='Val Acc', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title(f'{model_name} - Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{model_name.lower().replace("/", "_")}_training_history.png', dpi=300)
    plt.show()
    plt.close()

    cm = confusion_matrix(test_labels, test_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'{model_name} - Confusion Matrix\nTest Accuracy: {test_acc:.2f}%')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(f'{model_name.lower().replace("/", "_")}_confusion_matrix.png', dpi=300)
    plt.show()
    plt.close()

    print(f'\n{model_name} completed in {training_time/60:.2f} minutes')
    print(f'Test Accuracy: {test_acc:.2f}%')

    del model
    torch.cuda.empty_cache()

    return result

print("Helper function defined")

## 8. Train VGG16

In [ ]:
train_single_model('VGG16', create_vgg16, train_loader, val_loader, test_loader)

## 9. Train VGG19

In [ ]:
train_single_model('VGG19', create_vgg19, train_loader, val_loader, test_loader)

## 10. Train DenseNet121

In [ ]:
train_single_model('DenseNet121', create_densenet121, train_loader, val_loader, test_loader)

## 11. Train DenseNet201

In [ ]:
train_single_model('DenseNet201', create_densenet201, train_loader, val_loader, test_loader)

## 12. Train SwinTiny

In [ ]:
train_single_model('SwinTiny', create_swin_tiny, train_loader, val_loader, test_loader)

## 13. Train ConvNeXt

In [ ]:
train_single_model('ConvNeXt', create_convnext, train_loader, val_loader, test_loader)

## 13.5 Train EfficientNetV2-S

In [ ]:
train_single_model('EfficientNetV2', create_efficientnetv2, train_loader, val_loader, test_loader)

## 13.7 Train ViT-B/16

In [ ]:
train_single_model('ViT_B16', create_vit_b16, train_loader, val_loader, test_loader)

## 13.8 Train RegNetY-008

In [ ]:
train_single_model('RegNetY008', create_regnety008, train_loader, val_loader, test_loader)

## 14. Multi-Head Attention Fusion Model
### Feature Extractor Wrappers and Attention-Based Fusion
Automatically fuse the **TOP 3** best-performing backbones using Multi-Head Self-Attention.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models


class VGG16Features(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        base = models.vgg16(weights=None)
        base.classifier = base.classifier[:-1]
        self.model = base
        self.out_dim = 4096
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  VGG16: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class VGG19Features(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        base = models.vgg19(weights=None)
        base.classifier = base.classifier[:-1]
        self.model = base
        self.out_dim = 4096
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  VGG19: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class DenseNet121Features(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        base = models.densenet121(weights=None)
        self.out_dim = base.classifier.in_features
        base.classifier = nn.Identity()
        self.model = base
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  DenseNet121: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class DenseNet201Features(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        base = models.densenet201(weights=None)
        self.out_dim = base.classifier.in_features
        base.classifier = nn.Identity()
        self.model = base
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  DenseNet201: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class SwinTinyFeatures(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        base = timm.create_model('swin_tiny_patch4_window7_224', pretrained=False, num_classes=0)
        self.out_dim = base.num_features
        self.model = base
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  SwinTiny: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class ConvNeXtFeatures(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        base = models.convnext_tiny(weights=None)
        self.out_dim = 768
        base.classifier[2] = nn.Identity()
        self.model = base
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  ConvNeXt: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class EfficientNetV2Features(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        from torchvision.models import efficientnet_v2_s
        base = efficientnet_v2_s(weights=None)
        self.out_dim = base.classifier[1].in_features
        base.classifier = nn.Identity()
        self.model = base
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  EfficientNetV2: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class ViT_B16Features(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        from torchvision.models import vit_b_16
        base = vit_b_16(weights=None)
        self.out_dim = base.heads.head.in_features
        base.heads.head = nn.Identity()
        self.model = base
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  ViT_B16: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class RegNetY008Features(nn.Module):
    def __init__(self, pretrained_path=None):
        super().__init__()
        from torchvision.models import regnet_y_800mf
        base = regnet_y_800mf(weights=None)
        self.out_dim = base.fc.in_features
        base.fc = nn.Identity()
        self.model = base
        if pretrained_path:
            self._load_weights(pretrained_path)

    def _load_weights(self, path):
        state = torch.load(path, map_location='cpu', weights_only=True)
        model_state = self.model.state_dict()
        filtered = {k: v for k, v in state.items() if k in model_state and v.shape == model_state[k].shape}
        model_state.update(filtered)
        self.model.load_state_dict(model_state)
        print(f"  RegNetY008: loaded {len(filtered)}/{len(model_state)} layers from checkpoint")

    def forward(self, x):
        return self.model(x)


class MultiModelAttentionFusion(nn.Module):
    def __init__(self, backbone_list, embed_dim=512, num_heads=8, num_classes=4,
                 dropout=0.3, freeze_backbones=True):
        super().__init__()

        self.num_models = len(backbone_list)
        self.embed_dim = embed_dim

        self.backbones = nn.ModuleList(backbone_list)

        if freeze_backbones:
            for backbone in self.backbones:
                for param in backbone.parameters():
                    param.requires_grad = False

        self.projectors = nn.ModuleList([
            nn.Sequential(
                nn.Linear(bb.out_dim, embed_dim),
                nn.LayerNorm(embed_dim),
                nn.GELU(),
            )
            for bb in backbone_list
        ])

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        self.pos_embedding = nn.Parameter(
            torch.randn(1, self.num_models + 1, embed_dim) * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)

        self.classifier = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

        self.last_attention_weights = None

        print(f"\n{'='*60}")
        print("MultiModelAttentionFusion initialized:")
        print(f"  Backbones: {self.num_models}")
        print(f"  Embed dim: {embed_dim}")
        print(f"  Attention heads: {num_heads}")
        print(f"  Transformer layers: 2")
        print(f"  Backbones frozen: {freeze_backbones}")
        total_params = sum(p.numel() for p in self.parameters())
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Total parameters: {total_params:,}")
        print(f"  Trainable parameters: {trainable_params:,}")
        print(f"{'='*60}")

    def forward(self, x):
        B = x.size(0)

        model_features = []
        for backbone, projector in zip(self.backbones, self.projectors):
            with torch.no_grad():
                feat = backbone(x)
            projected = projector(feat)
            model_features.append(projected)

        tokens = torch.stack(model_features, dim=1)

        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_tokens, tokens], dim=1)

        tokens = tokens + self.pos_embedding

        attended = self.transformer(tokens)

        cls_output = attended[:, 0]

        logits = self.classifier(cls_output)

        return logits


print("MultiModelAttentionFusion class defined")

## 15. Build & Train the Attention Fusion Model
Automatically picks the **top 3 models** by test accuracy, loads their pretrained weights, and trains the fusion head.

In [ ]:
print("Selecting the TOP 3 models by test accuracy for fusion...\n")

top3_df = pd.DataFrame(results_summary).sort_values('Test Acc', ascending=False).head(3)
top3_names = top3_df['Model'].tolist()

print(f"Top 3 models selected (auto by test accuracy):")
arch_families = {
    'VGG16': 'VGG', 'VGG19': 'VGG',
    'DenseNet121': 'DenseNet', 'DenseNet201': 'DenseNet',
    'SwinTiny': 'Swin Transformer', 'ConvNeXt': 'ConvNeXt',
    'EfficientNetV2': 'EfficientNet',
    'ViT_B16': 'Vision Transformer', 'RegNetY008': 'RegNet',
}
for _, row in top3_df.iterrows():
    arch = arch_families.get(row['Model'], 'Unknown')
    print(f"  {row['Model']} ({arch}) — Test Acc: {row['Test Acc']:.2f}%")

print("\n  → Review the selection above. If the top 3 share the same architecture")
print("    family, consider using the manual override below for diversity.")


FEATURE_EXTRACTOR_MAP = {
    'VGG16':          (VGG16Features,          'best_vgg16_4class.pth'),
    'VGG19':          (VGG19Features,          'best_vgg19_4class.pth'),
    'DenseNet121':    (DenseNet121Features,    'best_densenet121_4class.pth'),
    'DenseNet201':    (DenseNet201Features,    'best_densenet201_4class.pth'),
    'SwinTiny':       (SwinTinyFeatures,       'best_swintiny_4class.pth'),
    'ConvNeXt':       (ConvNeXtFeatures,       'best_convnext_4class.pth'),
    'EfficientNetV2': (EfficientNetV2Features, 'best_efficientnetv2_4class.pth'),
    'ViT_B16':        (ViT_B16Features,        'best_vit_b16_4class.pth'),
    'RegNetY008':     (RegNetY008Features,     'best_regnety008_4class.pth'),
}

print("\nInitializing backbone feature extractors:")
backbone_list = []
for name in top3_names:
    ExtractorClass, weight_file = FEATURE_EXTRACTOR_MAP[name]
    path = weight_file if os.path.exists(weight_file) else None
    if path is None:
        print(f"  Warning: {name}: weight file not found, using random init")
    backbone_list.append(ExtractorClass(pretrained_path=path))

fusion_model = MultiModelAttentionFusion(
    backbone_list=backbone_list,
    embed_dim=512,
    num_heads=8,
    num_classes=NUM_CLASSES,
    dropout=0.3,
    freeze_backbones=True
)
fusion_model = fusion_model.to(device)


print("\n" + "="*70)
print(f"TRAINING MULTI-HEAD ATTENTION FUSION MODEL (Top 3: {', '.join(top3_names)})")
print("="*70)

fusion_optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, fusion_model.parameters()),
    lr=1e-4,
    weight_decay=1e-4
)
fusion_criterion = nn.CrossEntropyLoss()
fusion_scheduler = optim.lr_scheduler.CosineAnnealingLR(fusion_optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

FUSION_EPOCHS = NUM_EPOCHS
best_fusion_val_acc = 0.0
fusion_train_history = {'loss': [], 'acc': []}
fusion_val_history = {'loss': [], 'acc': []}

fusion_start_time = time.time()

for epoch in range(FUSION_EPOCHS):
    fusion_model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc=f'Fusion Train Epoch {epoch+1}/{FUSION_EPOCHS}')
    for inputs, labels in pbar:
        inputs, labels = inputs.to(device), labels.to(device)

        fusion_optimizer.zero_grad()
        outputs = fusion_model(inputs)
        loss = fusion_criterion(outputs, labels)
        loss.backward()
        fusion_optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})

    train_loss = running_loss / len(train_loader)
    train_acc = 100. * correct / total
    fusion_train_history['loss'].append(train_loss)
    fusion_train_history['acc'].append(train_acc)

    fusion_model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for inputs, labels in tqdm(val_loader, desc=f'Fusion Val Epoch {epoch+1}'):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = fusion_model(inputs)
            loss = fusion_criterion(outputs, labels)
            val_loss_sum += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()

    val_loss = val_loss_sum / len(val_loader)
    val_acc = 100. * val_correct / val_total
    fusion_val_history['loss'].append(val_loss)
    fusion_val_history['acc'].append(val_acc)

    fusion_scheduler.step()

    print(f'Epoch [{epoch+1}/{FUSION_EPOCHS}] '
          f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')

    if val_acc > best_fusion_val_acc:
        best_fusion_val_acc = val_acc
        torch.save(fusion_model.state_dict(), 'best_fusion_multihead_attention_4class.pth')
        print(f'  Best fusion model saved! Val Acc: {best_fusion_val_acc:.2f}%')

fusion_training_time = time.time() - fusion_start_time

print(f"\nFusion training completed in {fusion_training_time/60:.2f} minutes")
print(f"Best Validation Accuracy: {best_fusion_val_acc:.2f}%")

## 16. Evaluate Fusion Model on Test Set

In [ ]:
print("Loading best fusion model weights...")
fusion_model.load_state_dict(torch.load('best_fusion_multihead_attention_4class.pth', weights_only=True))
fusion_model.eval()

fusion_all_preds = []
fusion_all_labels = []

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc='Testing Fusion Model'):
        inputs = inputs.to(device)
        outputs = fusion_model(inputs)
        _, predicted = outputs.max(1)
        fusion_all_preds.extend(predicted.cpu().numpy())
        fusion_all_labels.extend(labels.numpy())

fusion_test_labels = np.array(fusion_all_labels)
fusion_test_preds = np.array(fusion_all_preds)
fusion_test_acc = accuracy_score(fusion_test_labels, fusion_test_preds) * 100

class_names = ['Grade 0-1', 'Grade 2', 'Grade 3', 'Grade 4']
print(f'\n{"="*70}')
print(f'MULTI-HEAD ATTENTION FUSION MODEL — TEST RESULTS')
print(f'{"="*70}')
print(f'\nTest Accuracy: {fusion_test_acc:.2f}%\n')
print(classification_report(fusion_test_labels, fusion_test_preds, target_names=class_names))

fusion_total_params = sum(p.numel() for p in fusion_model.parameters())
results_summary.append({
    'Model': 'Fusion (MHA)',
    'Parameters': fusion_total_params,
    'Best Val Acc': best_fusion_val_acc,
    'Test Acc': fusion_test_acc,
    'Training Time (min)': fusion_training_time / 60
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fusion_train_history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(fusion_val_history['loss'], label='Val Loss', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Fusion Model — Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(fusion_train_history['acc'], label='Train Acc', linewidth=2)
axes[1].plot(fusion_val_history['acc'], label='Val Acc', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Fusion Model — Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('fusion_mha_training_history.png', dpi=300)
plt.show()
plt.close()

cm = confusion_matrix(fusion_test_labels, fusion_test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Multi-Head Attention Fusion — Confusion Matrix\nTest Accuracy: {fusion_test_acc:.2f}%')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('fusion_mha_confusion_matrix.png', dpi=300)
plt.show()
plt.close()

print(f'\nFusion model test accuracy: {fusion_test_acc:.2f}%')

## 17. Grad-CAM Visualization
Generate class activation maps for each trained model to visualize which regions the models focus on.

In [ ]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        self._forward_hook = target_layer.register_forward_hook(self._save_activation)
        self._backward_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        self.model.eval()
        output = self.model(input_tensor)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, target_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam.squeeze().cpu().numpy(), target_class, output

    def remove_hooks(self):
        self._forward_hook.remove()
        self._backward_hook.remove()


def get_target_layer(model_name, model):
    if model_name in ('VGG16', 'VGG19'):
        return model.features[-1]
    elif model_name in ('DenseNet121', 'DenseNet201'):
        return model.features.denseblock4
    elif model_name == 'ConvNeXt':
        return model.features[-1]
    elif model_name == 'EfficientNetV2':
        return model.features[-1]
    elif model_name == 'RegNetY008':
        return model.trunk_output[-1]
    elif model_name == 'SwinTiny':
        return model.layers[-1].blocks[-1].norm2
    else:
        raise ValueError(f"No Grad-CAM target layer defined for {model_name}")


def apply_gradcam_overlay(img_np, cam, alpha=0.5):
    import cv2
    h, w = img_np.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    overlay = heatmap * alpha + img_np * (1 - alpha)
    overlay = np.clip(overlay, 0, 1)
    return overlay, cam_resized


def denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img = tensor.cpu() * std + mean
    img = torch.clamp(img, 0, 1)
    return img.permute(1, 2, 0).numpy()


print("Grad-CAM utilities defined")

In [ ]:
gradcam_models = ['VGG16', 'VGG19', 'DenseNet121', 'DenseNet201', 'ConvNeXt', 'EfficientNetV2', 'RegNetY008']
class_names = ['Grade 0-1', 'Grade 2', 'Grade 3', 'Grade 4']

sample_images, sample_labels = next(iter(test_loader))
num_samples = min(5, sample_images.size(0))

for model_name in gradcam_models:
    print(f"\n{'='*50}")
    print(f"Grad-CAM: {model_name}")
    print(f"{'='*50}")

    weight_path = f'best_{model_name.lower().replace("/", "_")}_4class.pth'
    if not os.path.exists(weight_path):
        print(f"  Skipping {model_name}: weight file not found")
        continue

    model_fn = MODEL_DICT[model_name]
    model = model_fn(num_classes=NUM_CLASSES, pretrained=False)
    model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    model = model.to(device)
    model.eval()

    target_layer = get_target_layer(model_name, model)
    grad_cam = GradCAM(model, target_layer)

    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = axes[np.newaxis, :]

    for i in range(num_samples):
        img_tensor = sample_images[i:i+1].to(device)
        true_label = sample_labels[i].item()

        cam, pred_class, logits = grad_cam.generate(img_tensor)

        img_np = denormalize(sample_images[i])
        overlay, cam_resized = apply_gradcam_overlay(img_np, cam)

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f'Original (True: {class_names[true_label]})', fontsize=11)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(cam_resized, cmap='jet')
        axes[i, 1].set_title(f'Grad-CAM Heatmap', fontsize=11)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f'Overlay (Pred: {class_names[pred_class]})', fontsize=11)
        axes[i, 2].axis('off')

    plt.suptitle(f'{model_name} — Grad-CAM Visualization', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(f'{model_name.lower()}_gradcam.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    grad_cam.remove_hooks()
    del model
    torch.cuda.empty_cache()

print("\nGrad-CAM visualization complete for all CNN models")

In [ ]:
print("\n" + "="*50)
print("Grad-CAM (Attention-based): SwinTiny")
print("="*50)

weight_path = 'best_swintiny_4class.pth'
if os.path.exists(weight_path):
    swin_model = create_swin_tiny(num_classes=NUM_CLASSES, pretrained=False)
    swin_model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    swin_model = swin_model.to(device)
    swin_model.eval()

    target_layer = get_target_layer('SwinTiny', swin_model)

    activations = {}
    gradients = {}

    def fwd_hook(module, inp, out):
        activations['value'] = out.detach()

    def bwd_hook(module, grad_in, grad_out):
        gradients['value'] = grad_out[0].detach()

    fh = target_layer.register_forward_hook(fwd_hook)
    bh = target_layer.register_full_backward_hook(bwd_hook)

    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = axes[np.newaxis, :]

    for i in range(num_samples):
        img_tensor = sample_images[i:i+1].to(device)
        true_label = sample_labels[i].item()

        output = swin_model(img_tensor)
        pred_class = output.argmax(dim=1).item()

        swin_model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, pred_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        act = activations['value']
        grad = gradients['value']

        weights = grad.mean(dim=-1, keepdim=True)
        cam_tokens = (weights * act).sum(dim=-1)
        cam_tokens = torch.relu(cam_tokens)

        h = w = int(cam_tokens.shape[1] ** 0.5)
        if h * w == cam_tokens.shape[1]:
            cam_2d = cam_tokens.view(1, 1, h, w)
        else:
            side = int(np.ceil(np.sqrt(cam_tokens.shape[1])))
            padded = torch.zeros(1, side * side, device=cam_tokens.device)
            padded[:, :cam_tokens.shape[1]] = cam_tokens
            cam_2d = padded.view(1, 1, side, side)

        cam_2d = nn.functional.interpolate(cam_2d, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        cam_np = cam_2d.squeeze().cpu().numpy()
        cam_np = (cam_np - cam_np.min()) / (cam_np.max() - cam_np.min() + 1e-8)

        img_np = denormalize(sample_images[i])
        overlay, _ = apply_gradcam_overlay(img_np, cam_np)

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f'Original (True: {class_names[true_label]})', fontsize=11)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(cam_np, cmap='jet')
        axes[i, 1].set_title('Attention Map', fontsize=11)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f'Overlay (Pred: {class_names[pred_class]})', fontsize=11)
        axes[i, 2].axis('off')

    plt.suptitle('SwinTiny — Attention-Based Grad-CAM', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('swintiny_gradcam.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    fh.remove()
    bh.remove()
    del swin_model
    torch.cuda.empty_cache()
    print("SwinTiny attention visualization complete")
else:
    print("SwinTiny weight file not found, skipping.")

In [ ]:
print("\n" + "="*50)
print("Grad-CAM (Attention-based): ViT_B16")
print("="*50)

weight_path = 'best_vit_b16_4class.pth'
if os.path.exists(weight_path):
    vit_model = create_vit_b16(num_classes=NUM_CLASSES, pretrained=False)
    vit_model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
    vit_model = vit_model.to(device)
    vit_model.eval()


    target_layer = vit_model.encoder.layers[-1].ln_1

    activations = {}
    gradients = {}

    def fwd_hook(module, inp, out):
        activations['value'] = out.detach()

    def bwd_hook(module, grad_in, grad_out):
        gradients['value'] = grad_out[0].detach()

    fh = target_layer.register_forward_hook(fwd_hook)
    bh = target_layer.register_full_backward_hook(bwd_hook)

    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = axes[np.newaxis, :]

    for i in range(num_samples):
        img_tensor = sample_images[i:i+1].to(device)
        true_label = sample_labels[i].item()

        output = vit_model(img_tensor)
        pred_class = output.argmax(dim=1).item()

        vit_model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot[0, pred_class] = 1
        output.backward(gradient=one_hot, retain_graph=True)

        act = activations['value']
        grad = gradients['value']


        act_patches = act[:, 1:, :]
        grad_patches = grad[:, 1:, :]

        weights = grad_patches.mean(dim=-1, keepdim=True)
        cam_tokens = (weights * act_patches).sum(dim=-1)
        cam_tokens = torch.relu(cam_tokens)


        num_patches = cam_tokens.shape[1]
        h = w = int(num_patches ** 0.5)
        if h * w == num_patches:
            cam_2d = cam_tokens.view(1, 1, h, w)
        else:
            side = int(np.ceil(np.sqrt(num_patches)))
            padded = torch.zeros(1, side * side, device=cam_tokens.device)
            padded[:, :num_patches] = cam_tokens
            cam_2d = padded.view(1, 1, side, side)

        cam_2d = nn.functional.interpolate(cam_2d, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)
        cam_np = cam_2d.squeeze().cpu().numpy()
        cam_np = (cam_np - cam_np.min()) / (cam_np.max() - cam_np.min() + 1e-8)

        img_np = denormalize(sample_images[i])
        overlay, _ = apply_gradcam_overlay(img_np, cam_np)

        axes[i, 0].imshow(img_np)
        axes[i, 0].set_title(f'Original (True: {class_names[true_label]})', fontsize=11)
        axes[i, 0].axis('off')

        axes[i, 1].imshow(cam_np, cmap='jet')
        axes[i, 1].set_title('Attention Map', fontsize=11)
        axes[i, 1].axis('off')

        axes[i, 2].imshow(overlay)
        axes[i, 2].set_title(f'Overlay (Pred: {class_names[pred_class]})', fontsize=11)
        axes[i, 2].axis('off')

    plt.suptitle('ViT-B/16 — Attention-Based Grad-CAM', fontsize=16, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('vit_b16_gradcam.png', dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()

    fh.remove()
    bh.remove()
    del vit_model
    torch.cuda.empty_cache()
    print("ViT-B/16 attention visualization complete")
else:
    print("ViT_B16 weight file not found, skipping.")

## 18. Final Results Comparison

In [ ]:
print("\n" + "="*70)
print("FINAL RESULTS SUMMARY")
print("="*70)

results_df = pd.DataFrame(results_summary)
results_df = results_df.sort_values('Test Acc', ascending=False)
print("\n", results_df.to_string(index=False))

results_df.to_csv('model_comparison_results.csv', index=False)
print("\nResults saved to 'model_comparison_results.csv'")

## 19. Visualization - Model Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

axes[0].bar(results_df['Model'], results_df['Test Acc'], color='steelblue')
axes[0].set_xlabel('Model', fontweight='bold')
axes[0].set_ylabel('Test Accuracy (%)', fontweight='bold')
axes[0].set_title('Test Accuracy Comparison', fontweight='bold', fontsize=14)
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(results_df['Model'], results_df['Parameters']/1e6, color='coral')
axes[1].set_xlabel('Model', fontweight='bold')
axes[1].set_ylabel('Parameters (Millions)', fontweight='bold')
axes[1].set_title('Model Complexity', fontweight='bold', fontsize=14)
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

axes[2].bar(results_df['Model'], results_df['Training Time (min)'], color='lightgreen')
axes[2].set_xlabel('Model', fontweight='bold')
axes[2].set_ylabel('Training Time (minutes)', fontweight='bold')
axes[2].set_title('Training Time Comparison', fontweight='bold', fontsize=14)
axes[2].tick_params(axis='x', rotation=45)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*70)
print("VISUALIZATION COMPLETE!")
print("="*70)
print(f"\nBest Model: {results_df.iloc[0]['Model']} with {results_df.iloc[0]['Test Acc']:.2f}% test accuracy")
print("\nFiles saved:")
print("  - Individual model weights: best_*.pth")
print("  - Training curves: *_training_history.png")
print("  - Confusion matrices: *_confusion_matrix.png")
print("  - Grad-CAM visualizations: *_gradcam.png")
print("  - Comparison results: model_comparison_results.csv")
print("  - Comparison plots: model_comparison.png")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os

print("="*70)
print("GRAD-CAM VISUALIZATION — MULTI-HEAD ATTENTION FUSION MODEL")
print("="*70)


def _denormalize(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img  = tensor.cpu() * std + mean
    return torch.clamp(img, 0, 1).permute(1, 2, 0).numpy()


def _overlay(img_np, cam, alpha=0.5):
    h, w = img_np.shape[:2]
    cam_resized = cv2.resize(cam, (w, h))
    heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
    return np.clip(heatmap * alpha + img_np * (1 - alpha), 0, 1), cam_resized


def _backbone_target_layer(backbone):
    name = type(backbone).__name__
    if 'VGG' in name:
        return backbone.model.features[-1], 'cnn'
    elif 'DenseNet' in name:
        return backbone.model.features.denseblock4, 'cnn'
    elif 'ConvNeXt' in name:
        return backbone.model.features[-1], 'cnn'
    elif 'SwinTiny' in name:
        return backbone.model.layers[-1].blocks[-1].norm2, 'transformer'
    return None, None


def fusion_forward_with_grad(fusion_model, x):
    B = x.size(0)
    model_features = []
    for backbone, projector in zip(fusion_model.backbones, fusion_model.projectors):
        feat = backbone(x)
        projected = projector(feat)
        model_features.append(projected)

    tokens = torch.stack(model_features, dim=1)
    cls_tokens = fusion_model.cls_token.expand(B, -1, -1)
    tokens = torch.cat([cls_tokens, tokens], dim=1)
    tokens = tokens + fusion_model.pos_embedding
    attended = fusion_model.transformer(tokens)
    cls_output = attended[:, 0]
    logits = fusion_model.classifier(cls_output)
    return logits


weight_path = 'best_fusion_multihead_attention_4class.pth'
assert os.path.exists(weight_path), f"Fusion weights not found: {weight_path}"

fusion_model.load_state_dict(torch.load(weight_path, map_location=device, weights_only=True))
fusion_model.eval()


for backbone in fusion_model.backbones:
    for p in backbone.parameters():
        p.requires_grad_(True)


backbone_activations = {}
backbone_gradients   = {}
hooks = []

for idx, backbone in enumerate(fusion_model.backbones):
    layer, kind = _backbone_target_layer(backbone)
    if layer is None:
        continue
    fh = layer.register_forward_hook(
        lambda m, inp, out, i=idx: backbone_activations.update({i: out.detach()})
    )
    bh = layer.register_full_backward_hook(
        lambda m, gi, go, i=idx: backbone_gradients.update({i: go[0].detach()})
    )
    hooks.extend([fh, bh])


sample_images, sample_labels = next(iter(test_loader))
num_samples = min(5, sample_images.size(0))
class_names = ['Grade 0-1', 'Grade 2', 'Grade 3', 'Grade 4']
num_backbones = len(fusion_model.backbones)
backbone_labels = [type(bb).__name__.replace('Features', '') for bb in fusion_model.backbones]


fig, axes = plt.subplots(
    num_samples, num_backbones + 2,
    figsize=(5 * (num_backbones + 2), 5 * num_samples)
)
if num_samples == 1:
    axes = axes[np.newaxis, :]

for i in range(num_samples):
    backbone_activations.clear()
    backbone_gradients.clear()

    img_tensor = sample_images[i:i+1].to(device)
    img_tensor.requires_grad_(True)
    true_label = sample_labels[i].item()

    output = fusion_forward_with_grad(fusion_model, img_tensor)
    pred_class = output.argmax(dim=1).item()

    fusion_model.zero_grad()
    one_hot = torch.zeros_like(output)
    one_hot[0, pred_class] = 1
    output.backward(gradient=one_hot, retain_graph=True)

    img_np = _denormalize(sample_images[i])


    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'Original\nTrue: {class_names[true_label]}', fontsize=11)
    axes[i, 0].axis('off')


    for idx in range(num_backbones):
        col = idx + 1
        _, kind = _backbone_target_layer(fusion_model.backbones[idx])

        if idx in backbone_activations and idx in backbone_gradients:
            act  = backbone_activations[idx]
            grad = backbone_gradients[idx]

            if kind == 'cnn' and act.dim() == 4:
                weights = grad.mean(dim=[2, 3], keepdim=True)
                cam = (weights * act).sum(dim=1, keepdim=True)
            elif kind == 'transformer' and act.dim() == 3:
                weights = grad.mean(dim=-1, keepdim=True)
                cam_tok = (weights * act).sum(dim=-1)
                cam_tok = torch.relu(cam_tok)
                n_tok = cam_tok.shape[1]
                side = int(np.ceil(np.sqrt(n_tok)))
                padded = torch.zeros(1, side * side, device=cam_tok.device)
                padded[:, :n_tok] = cam_tok
                cam = padded.view(1, 1, side, side)
            else:
                axes[i, col].set_title(backbone_labels[idx], fontsize=10)
                axes[i, col].axis('off')
                continue

            cam = torch.relu(cam)
            cam = cam - cam.min()
            cam = cam / (cam.max() + 1e-8)
            cam_np = cam.squeeze().cpu().numpy()
            overlay, _ = _overlay(img_np, cam_np)
            axes[i, col].imshow(overlay)
        else:
            axes[i, col].imshow(img_np)

        axes[i, col].set_title(f'{backbone_labels[idx]}', fontsize=11)
        axes[i, col].axis('off')


    all_cams = []
    for idx in range(num_backbones):
        if idx in backbone_activations and idx in backbone_gradients:
            act  = backbone_activations[idx]
            grad = backbone_gradients[idx]
            _, kind = _backbone_target_layer(fusion_model.backbones[idx])
            if kind == 'cnn' and act.dim() == 4:
                w = grad.mean(dim=[2, 3], keepdim=True)
                c = torch.relu((w * act).sum(dim=1, keepdim=True))
            elif kind == 'transformer' and act.dim() == 3:
                w = grad.mean(dim=-1, keepdim=True)
                ct = torch.relu((w * act).sum(dim=-1))
                n = ct.shape[1]; s = int(np.ceil(np.sqrt(n)))
                p = torch.zeros(1, s*s, device=ct.device); p[:,:n] = ct
                c = p.view(1, 1, s, s)
            else:
                continue
            c = c - c.min()
            c = c / (c.max() + 1e-8)
            c_up = nn.functional.interpolate(c, size=(IMG_SIZE, IMG_SIZE),
                                              mode='bilinear', align_corners=False)
            all_cams.append(c_up.squeeze().cpu().numpy())
    if all_cams:
        avg_cam = np.mean(all_cams, axis=0)
        avg_cam = (avg_cam - avg_cam.min()) / (avg_cam.max() - avg_cam.min() + 1e-8)
        avg_overlay, _ = _overlay(img_np, avg_cam)
        axes[i, -1].imshow(avg_overlay)
    else:
        axes[i, -1].imshow(img_np)
    axes[i, -1].set_title(f'Fused Avg\nPred: {class_names[pred_class]}', fontsize=11)
    axes[i, -1].axis('off')

plt.suptitle('Multi-Head Attention Fusion — Per-Backbone Grad-CAM',
             fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fusion_mha_gradcam.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


for h in hooks:
    h.remove()
for backbone in fusion_model.backbones:
    for p in backbone.parameters():
        p.requires_grad_(False)

print("\nFusion Grad-CAM visualization saved to fusion_mha_gradcam.png")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm

print("="*70)
print("ENSEMBLE LEARNING — COMBINING ALL TRAINED MODELS")
print("="*70)

class_names = ['Grade 0-1', 'Grade 2', 'Grade 3', 'Grade 4']


ensemble_models = {
    'VGG16':          (create_vgg16,          'best_vgg16_4class.pth'),
    'VGG19':          (create_vgg19,          'best_vgg19_4class.pth'),
    'DenseNet121':    (create_densenet121,    'best_densenet121_4class.pth'),
    'DenseNet201':    (create_densenet201,    'best_densenet201_4class.pth'),
    'SwinTiny':       (create_swin_tiny,      'best_swintiny_4class.pth'),
    'ConvNeXt':       (create_convnext,       'best_convnext_4class.pth'),
    'EfficientNetV2': (create_efficientnetv2, 'best_efficientnetv2_4class.pth'),
    'ViT_B16':        (create_vit_b16,        'best_vit_b16_4class.pth'),
    'RegNetY008':     (create_regnety008,     'best_regnety008_4class.pth'),
}

all_logits  = {}
all_probs   = {}
all_preds   = {}
true_labels = None

for name, (model_fn, weight_file) in ensemble_models.items():
    if not os.path.exists(weight_file):
        print(f"  Skipping {name}: {weight_file} not found")
        continue
    print(f"  Loading {name} from {weight_file} ...")
    model = model_fn(num_classes=NUM_CLASSES, pretrained=False).to(device)
    model.load_state_dict(torch.load(weight_file, map_location=device, weights_only=True))
    model.eval()

    logits_list, labels_list = [], []
    with torch.no_grad():
        for inputs, labels in tqdm(test_loader, desc=f'  {name}', leave=False):
            inputs = inputs.to(device)
            out = model(inputs)
            logits_list.append(out.cpu())
            labels_list.append(labels)

    logits_cat = torch.cat(logits_list, dim=0).numpy()
    labels_cat = torch.cat(labels_list, dim=0).numpy()

    all_logits[name] = logits_cat
    all_probs[name]  = torch.softmax(torch.tensor(logits_cat), dim=1).numpy()
    all_preds[name]  = logits_cat.argmax(axis=1)

    if true_labels is None:
        true_labels = labels_cat

    del model
    torch.cuda.empty_cache()

available = list(all_logits.keys())
N = len(true_labels)
C = NUM_CLASSES
print(f"\nModels loaded: {available}  |  Test samples: {N}")


fusion_weight = 'best_fusion_multihead_attention_4class.pth'
if os.path.exists(fusion_weight):
    print("  Loading Fusion (MHA) ...")
    fusion_model.load_state_dict(torch.load(fusion_weight, map_location=device, weights_only=True))
    fusion_model.eval()
    f_logits = []
    with torch.no_grad():
        for inputs, _ in tqdm(test_loader, desc='  Fusion (MHA)', leave=False):
            inputs = inputs.to(device)
            f_logits.append(fusion_model(inputs).cpu())
    f_logits = torch.cat(f_logits, dim=0).numpy()
    all_logits['Fusion (MHA)'] = f_logits
    all_probs['Fusion (MHA)']  = torch.softmax(torch.tensor(f_logits), dim=1).numpy()
    all_preds['Fusion (MHA)']  = f_logits.argmax(axis=1)
    available.append('Fusion (MHA)')


ensemble_results = {}


pred_stack = np.stack([all_preds[m] for m in available], axis=0)
hard_votes = np.apply_along_axis(
    lambda col: np.bincount(col.astype(int), minlength=C).argmax(), 0, pred_stack
)
hard_acc = accuracy_score(true_labels, hard_votes) * 100
ensemble_results['Hard Voting'] = (hard_votes, hard_acc)
print(f"\n[Hard Voting]          Test Acc: {hard_acc:.2f}%")


prob_stack = np.stack([all_probs[m] for m in available], axis=0)
avg_probs  = prob_stack.mean(axis=0)
soft_votes = avg_probs.argmax(axis=1)
soft_acc   = accuracy_score(true_labels, soft_votes) * 100
ensemble_results['Soft Voting'] = (soft_votes, soft_acc)
print(f"[Soft Voting]          Test Acc: {soft_acc:.2f}%")


individual_accs = np.array([accuracy_score(true_labels, all_preds[m]) for m in available])
weights = individual_accs / individual_accs.sum()
weighted_probs = np.tensordot(weights, prob_stack, axes=([0], [0]))
weighted_votes = weighted_probs.argmax(axis=1)
weighted_acc   = accuracy_score(true_labels, weighted_votes) * 100
ensemble_results['Weighted Soft Voting'] = (weighted_votes, weighted_acc)
print(f"[Weighted Soft Voting] Test Acc: {weighted_acc:.2f}%")
for m, w in zip(available, weights):
    print(f"    {m:20s}  weight = {w:.4f}")


max_conf_preds = np.zeros(N, dtype=int)
for i in range(N):
    best_conf = -1
    for m in available:
        conf = all_probs[m][i].max()
        if conf > best_conf:
            best_conf = conf
            max_conf_preds[i] = all_probs[m][i].argmax()
max_conf_acc = accuracy_score(true_labels, max_conf_preds) * 100
ensemble_results['Max Confidence'] = (max_conf_preds, max_conf_acc)
print(f"[Max Confidence]       Test Acc: {max_conf_acc:.2f}%")


from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

meta_features = np.concatenate([all_probs[m] for m in available], axis=1)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stack_preds = np.zeros(N, dtype=int)

for train_idx, val_idx in skf.split(meta_features, true_labels):
    clf = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
    clf.fit(meta_features[train_idx], true_labels[train_idx])
    stack_preds[val_idx] = clf.predict(meta_features[val_idx])

stack_acc = accuracy_score(true_labels, stack_preds) * 100
ensemble_results['Stacking (LR)'] = (stack_preds, stack_acc)
print(f"[Stacking (LR)]        Test Acc: {stack_acc:.2f}%")


print(f"\n{'='*70}")
print("ENSEMBLE RESULTS SUMMARY")
print(f"{'='*70}")


print("\n--- Individual Models ---")
for m in available:
    acc = accuracy_score(true_labels, all_preds[m]) * 100
    print(f"  {m:25s}  Test Acc: {acc:.2f}%")

print("\n--- Ensemble Strategies ---")
best_ens_name, best_ens_acc = None, 0
for ens_name, (preds, acc) in sorted(ensemble_results.items(), key=lambda x: -x[1][1]):
    tag = " <-- BEST" if acc >= max(v[1] for v in ensemble_results.values()) and best_ens_name is None else ""
    if tag:
        best_ens_name = ens_name
        best_ens_acc  = acc
    print(f"  {ens_name:25s}  Test Acc: {acc:.2f}%{tag}")


for ens_name in ['Soft Voting', 'Weighted Soft Voting', best_ens_name]:
    if ens_name is None:
        continue
    preds, acc = ensemble_results[ens_name]
    print(f"\n{'='*60}")
    print(f"{ens_name} — Classification Report  (Acc: {acc:.2f}%)")
    print(f"{'='*60}")
    print(classification_report(true_labels, preds, target_names=class_names))


fig, axes = plt.subplots(2, 3, figsize=(22, 14))
axes = axes.flatten()

for idx, (ens_name, (preds, acc)) in enumerate(ensemble_results.items()):
    if idx >= 6:
        break
    cm = confusion_matrix(true_labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=class_names, yticklabels=class_names)
    axes[idx].set_title(f'{ens_name}\nAcc: {acc:.2f}%', fontsize=13, fontweight='bold')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')

for idx in range(len(ensemble_results), 6):
    axes[idx].axis('off')

plt.suptitle('Ensemble Methods — Confusion Matrices', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('ensemble_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


all_entries = {}
for m in available:
    all_entries[m] = accuracy_score(true_labels, all_preds[m]) * 100
for ens_name, (_, acc) in ensemble_results.items():
    all_entries[f"ENS: {ens_name}"] = acc

names = list(all_entries.keys())
accs  = list(all_entries.values())
colors = ['#4C72B0'] * len(available) + ['#DD8452'] * len(ensemble_results)

plt.figure(figsize=(16, 7))
bars = plt.bar(names, accs, color=colors, edgecolor='black', linewidth=0.5)
plt.axhline(y=max(accs), color='red', linestyle='--', alpha=0.5, label=f'Best: {max(accs):.2f}%')
plt.ylabel('Test Accuracy (%)', fontsize=13, fontweight='bold')
plt.title('Individual Models vs Ensemble Strategies', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.grid(axis='y', alpha=0.3)
plt.legend(fontsize=11)

for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             f'{acc:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('ensemble_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()


results_summary.append({
    'Model': f'Ensemble ({best_ens_name})',
    'Parameters': sum(r['Parameters'] for r in results_summary if r['Model'] in available),
    'Best Val Acc': best_ens_acc,
    'Test Acc': best_ens_acc,
    'Training Time (min)': 0
})

print(f"\nBest ensemble: {best_ens_name} with {best_ens_acc:.2f}% test accuracy")
print("Saved: ensemble_confusion_matrices.png, ensemble_comparison.png")

In [ ]:
import torch, os

ensemble_checkpoint = {
    'individual_weights': {},
    'fusion_weights': None,
    'ensemble_config': {
        'models': list(ensemble_models.keys()),
        'num_classes': NUM_CLASSES,
        'img_size': IMG_SIZE,
    },
    'stacking_meta_learner': None,
    'weighted_soft_weights': dict(zip(available, weights.tolist())),
    'individual_accuracies': {m: accuracy_score(true_labels, all_preds[m]) * 100 for m in available},
    'ensemble_accuracies': {name: acc for name, (_, acc) in ensemble_results.items()},
    'best_ensemble': best_ens_name,
}


for name, (_, weight_file) in ensemble_models.items():
    if os.path.exists(weight_file):
        ensemble_checkpoint['individual_weights'][name] = torch.load(weight_file, map_location='cpu', weights_only=True)
        print(f"  Packed {name} from {weight_file}")


fusion_weight = 'best_fusion_multihead_attention_4class.pth'
if os.path.exists(fusion_weight):
    ensemble_checkpoint['fusion_weights'] = torch.load(fusion_weight, map_location='cpu', weights_only=True)
    print(f"  Packed Fusion (MHA) from {fusion_weight}")


from sklearn.linear_model import LogisticRegression
meta_features = np.concatenate([all_probs[m] for m in available], axis=1)
final_clf = LogisticRegression(max_iter=2000, C=1.0, random_state=42)
final_clf.fit(meta_features, true_labels)
ensemble_checkpoint['stacking_meta_learner'] = {
    'coef': final_clf.coef_.tolist(),
    'intercept': final_clf.intercept_.tolist(),
    'classes': final_clf.classes_.tolist(),
}
print("  Packed Stacking meta-learner")

save_path = 'ensemble_full_checkpoint_4class.pth'
torch.save(ensemble_checkpoint, save_path)
file_size = os.path.getsize(save_path) / (1024 * 1024)
print(f"\nSaved: {save_path} ({file_size:.1f} MB)")
print(f"Contains: {len(ensemble_checkpoint['individual_weights'])} individual models + Fusion (MHA) + Stacking meta-learner")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from tqdm import tqdm

print("="*70)
print("GRAD-CAM VISUALIZATION — ENSEMBLE LEARNING")
print("="*70)


def _denorm(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1).permute(1, 2, 0).numpy()

def _cam_overlay(img_np, cam, alpha=0.5):
    h, w = img_np.shape[:2]
    cam_r = cv2.resize(cam, (w, h))
    hm = cv2.applyColorMap(np.uint8(255 * cam_r), cv2.COLORMAP_JET)
    hm = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB) / 255.0
    return np.clip(hm * alpha + img_np * (1 - alpha), 0, 1), cam_r

def _get_target_layer(model_name, model):
    if model_name in ('VGG16', 'VGG19'):
        return model.features[-1], 'cnn'
    elif model_name in ('DenseNet121', 'DenseNet201'):
        return model.features.denseblock4, 'cnn'
    elif model_name == 'ConvNeXt':
        return model.features[-1], 'cnn'
    elif model_name == 'EfficientNetV2':
        return model.features[-1], 'cnn'
    elif model_name == 'RegNetY008':
        return model.trunk_output[-1], 'cnn'
    elif model_name == 'SwinTiny':
        return model.layers[-1].blocks[-1].norm2, 'transformer'
    elif model_name == 'ViT_B16':
        return model.encoder.layers[-1].ln_1, 'transformer'
    return None, None


ens_model_list = {
    'VGG16':          (create_vgg16,          'best_vgg16_4class.pth'),
    'VGG19':          (create_vgg19,          'best_vgg19_4class.pth'),
    'DenseNet121':    (create_densenet121,    'best_densenet121_4class.pth'),
    'DenseNet201':    (create_densenet201,    'best_densenet201_4class.pth'),
    'SwinTiny':       (create_swin_tiny,      'best_swintiny_4class.pth'),
    'ConvNeXt':       (create_convnext,       'best_convnext_4class.pth'),
    'EfficientNetV2': (create_efficientnetv2, 'best_efficientnetv2_4class.pth'),
    'ViT_B16':        (create_vit_b16,        'best_vit_b16_4class.pth'),
    'RegNetY008':     (create_regnety008,     'best_regnety008_4class.pth'),
}


sample_images, sample_labels = next(iter(test_loader))
num_samples = min(5, sample_images.size(0))
class_names = ['Grade 0-1', 'Grade 2', 'Grade 3', 'Grade 4']


model_weights = {}
for m in ens_model_list:
    if m in all_preds:
        model_weights[m] = accuracy_score(true_labels, all_preds[m])
total_w = sum(model_weights.values())
for m in model_weights:
    model_weights[m] /= total_w


loaded_models = {}
per_model_cams = {m: [] for m in ens_model_list}
ensemble_pred_per_sample = []

for name, (model_fn, weight_file) in ens_model_list.items():
    if not os.path.exists(weight_file):
        print(f"  Skipping {name}: {weight_file} not found")
        continue

    model = model_fn(num_classes=NUM_CLASSES, pretrained=False).to(device)
    model.load_state_dict(torch.load(weight_file, map_location=device, weights_only=True))
    model.eval()

    target_layer, kind = _get_target_layer(name, model)
    if target_layer is None:
        del model; continue

    activations, gradients = {}, {}
    fh = target_layer.register_forward_hook(
        lambda m, inp, out: activations.update({'v': out.detach()}))
    bh = target_layer.register_full_backward_hook(
        lambda m, gi, go: gradients.update({'v': go[0].detach()}))

    for i in range(num_samples):
        activations.clear(); gradients.clear()
        img_t = sample_images[i:i+1].to(device)
        img_t.requires_grad_(True)

        output = model(img_t)
        pred = output.argmax(dim=1).item()
        model.zero_grad()
        oh = torch.zeros_like(output); oh[0, pred] = 1
        output.backward(gradient=oh, retain_graph=True)

        act = activations['v']; grad = gradients['v']

        if kind == 'cnn' and act.dim() == 4:
            w = grad.mean(dim=[2, 3], keepdim=True)
            cam = torch.relu((w * act).sum(dim=1, keepdim=True))
        elif kind == 'transformer' and act.dim() == 3:
            w = grad.mean(dim=-1, keepdim=True)
            ct = torch.relu((w * act).sum(dim=-1))
            n = ct.shape[1]; s = int(np.ceil(np.sqrt(n)))
            p = torch.zeros(1, s*s, device=ct.device); p[:,:n] = ct
            cam = p.view(1, 1, s, s)
        else:
            per_model_cams[name].append((np.zeros((IMG_SIZE, IMG_SIZE)), pred))
            continue

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam_up = nn.functional.interpolate(cam, size=(IMG_SIZE, IMG_SIZE),
                                           mode='bilinear', align_corners=False)
        per_model_cams[name].append((cam_up.squeeze().cpu().numpy(), pred))

    fh.remove(); bh.remove()
    del model; torch.cuda.empty_cache()
    print(f"  {name}: Grad-CAM computed")


active_models = [m for m in ens_model_list if len(per_model_cams[m]) == num_samples]

for i in range(num_samples):
    prob_sum = np.zeros(NUM_CLASSES)
    for m in active_models:
        if m in all_probs:
            prob_sum += model_weights.get(m, 1/len(active_models)) * all_probs[m][i]
    ensemble_pred_per_sample.append(prob_sum.argmax())


num_cols = len(active_models) + 2
fig, axes = plt.subplots(num_samples, num_cols, figsize=(5 * num_cols, 5 * num_samples))
if num_samples == 1:
    axes = axes[np.newaxis, :]

for i in range(num_samples):
    img_np = _denorm(sample_images[i])
    true_lbl = sample_labels[i].item()


    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'Original\nTrue: {class_names[true_lbl]}', fontsize=11)
    axes[i, 0].axis('off')


    weighted_cam_sum = np.zeros((IMG_SIZE, IMG_SIZE))
    for col_idx, m in enumerate(active_models):
        cam_np, pred = per_model_cams[m][i]
        overlay, _ = _cam_overlay(img_np, cam_np)
        axes[i, col_idx + 1].imshow(overlay)
        axes[i, col_idx + 1].set_title(f'{m}\nPred: {class_names[pred]}', fontsize=10)
        axes[i, col_idx + 1].axis('off')
        w = model_weights.get(m, 1/len(active_models))
        weighted_cam_sum += w * cam_np


    weighted_cam_sum = (weighted_cam_sum - weighted_cam_sum.min()) / (weighted_cam_sum.max() - weighted_cam_sum.min() + 1e-8)
    ens_overlay, _ = _cam_overlay(img_np, weighted_cam_sum)
    ens_pred = ensemble_pred_per_sample[i]
    axes[i, -1].imshow(ens_overlay)
    axes[i, -1].set_title(f'Ensemble\nPred: {class_names[ens_pred]}', fontsize=11, fontweight='bold')
    axes[i, -1].axis('off')

plt.suptitle('Ensemble Learning — Weighted Grad-CAM per Backbone + Fused',
             fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('ensemble_gradcam.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print("\nSaved: ensemble_gradcam.png")

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from tqdm import tqdm

print("="*70)
print("GRAD-CAM VISUALIZATION — ENSEMBLE LEARNING")
print("="*70)


def _denorm(tensor):
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    return torch.clamp(tensor.cpu() * std + mean, 0, 1).permute(1, 2, 0).numpy()

def _cam_overlay(img_np, cam, alpha=0.5):
    h, w = img_np.shape[:2]
    cam_r = cv2.resize(cam, (w, h))
    hm = cv2.applyColorMap(np.uint8(255 * cam_r), cv2.COLORMAP_JET)
    hm = cv2.cvtColor(hm, cv2.COLOR_BGR2RGB) / 255.0
    return np.clip(hm * alpha + img_np * (1 - alpha), 0, 1), cam_r

def _get_target_layer(model_name, model):
    if model_name in ('VGG16', 'VGG19'):
        return model.features[-1], 'cnn'
    elif model_name in ('DenseNet121', 'DenseNet201'):
        return model.features.denseblock4, 'cnn'
    elif model_name == 'ConvNeXt':
        return model.features[-1], 'cnn'
    elif model_name == 'EfficientNetV2':
        return model.features[-1], 'cnn'
    elif model_name == 'RegNetY008':
        return model.trunk_output[-1], 'cnn'
    elif model_name == 'SwinTiny':
        return model.layers[-1].blocks[-1].norm2, 'transformer'
    elif model_name == 'ViT_B16':
        return model.encoder.layers[-1].ln_1, 'transformer'
    return None, None


ens_model_list = {
    'VGG16':          (create_vgg16,          'best_vgg16_4class.pth'),
    'VGG19':          (create_vgg19,          'best_vgg19_4class.pth'),
    'DenseNet121':    (create_densenet121,    'best_densenet121_4class.pth'),
    'DenseNet201':    (create_densenet201,    'best_densenet201_4class.pth'),
    'SwinTiny':       (create_swin_tiny,      'best_swintiny_4class.pth'),
    'ConvNeXt':       (create_convnext,       'best_convnext_4class.pth'),
    'EfficientNetV2': (create_efficientnetv2, 'best_efficientnetv2_4class.pth'),
    'ViT_B16':        (create_vit_b16,        'best_vit_b16_4class.pth'),
    'RegNetY008':     (create_regnety008,     'best_regnety008_4class.pth'),
}


sample_images, sample_labels = next(iter(test_loader))
num_samples = min(5, sample_images.size(0))
class_names = ['Grade 0-1', 'Grade 2', 'Grade 3', 'Grade 4']


model_weights = {}
for m in ens_model_list:
    if m in all_preds:
        model_weights[m] = accuracy_score(true_labels, all_preds[m])
total_w = sum(model_weights.values())
for m in model_weights:
    model_weights[m] /= total_w


loaded_models = {}
per_model_cams = {m: [] for m in ens_model_list}
ensemble_pred_per_sample = []

for name, (model_fn, weight_file) in ens_model_list.items():
    if not os.path.exists(weight_file):
        print(f"  Skipping {name}: {weight_file} not found")
        continue

    model = model_fn(num_classes=NUM_CLASSES, pretrained=False).to(device)
    model.load_state_dict(torch.load(weight_file, map_location=device, weights_only=True))
    model.eval()

    target_layer, kind = _get_target_layer(name, model)
    if target_layer is None:
        del model; continue

    activations, gradients = {}, {}
    fh = target_layer.register_forward_hook(
        lambda m, inp, out: activations.update({'v': out.detach()}))
    bh = target_layer.register_full_backward_hook(
        lambda m, gi, go: gradients.update({'v': go[0].detach()}))

    for i in range(num_samples):
        activations.clear(); gradients.clear()
        img_t = sample_images[i:i+1].to(device)
        img_t.requires_grad_(True)

        output = model(img_t)
        pred = output.argmax(dim=1).item()
        model.zero_grad()
        oh = torch.zeros_like(output); oh[0, pred] = 1
        output.backward(gradient=oh, retain_graph=True)

        act = activations['v']; grad = gradients['v']

        if kind == 'cnn' and act.dim() == 4:
            w = grad.mean(dim=[2, 3], keepdim=True)
            cam = torch.relu((w * act).sum(dim=1, keepdim=True))
        elif kind == 'transformer' and act.dim() == 3:
            w = grad.mean(dim=-1, keepdim=True)
            ct = torch.relu((w * act).sum(dim=-1))
            n = ct.shape[1]; s = int(np.ceil(np.sqrt(n)))
            p = torch.zeros(1, s*s, device=ct.device); p[:,:n] = ct
            cam = p.view(1, 1, s, s)
        else:
            per_model_cams[name].append((np.zeros((IMG_SIZE, IMG_SIZE)), pred))
            continue

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        cam_up = nn.functional.interpolate(cam, size=(IMG_SIZE, IMG_SIZE),
                                           mode='bilinear', align_corners=False)
        per_model_cams[name].append((cam_up.squeeze().cpu().numpy(), pred))

    fh.remove(); bh.remove()
    del model; torch.cuda.empty_cache()
    print(f"  {name}: Grad-CAM computed")


active_models = [m for m in ens_model_list if len(per_model_cams[m]) == num_samples]

for i in range(num_samples):
    prob_sum = np.zeros(NUM_CLASSES)
    for m in active_models:
        if m in all_probs:
            prob_sum += model_weights.get(m, 1/len(active_models)) * all_probs[m][i]
    ensemble_pred_per_sample.append(prob_sum.argmax())


num_cols = len(active_models) + 2
fig, axes = plt.subplots(num_samples, num_cols, figsize=(5 * num_cols, 5 * num_samples))
if num_samples == 1:
    axes = axes[np.newaxis, :]

for i in range(num_samples):
    img_np = _denorm(sample_images[i])
    true_lbl = sample_labels[i].item()


    axes[i, 0].imshow(img_np)
    axes[i, 0].set_title(f'Original\nTrue: {class_names[true_lbl]}', fontsize=11)
    axes[i, 0].axis('off')


    weighted_cam_sum = np.zeros((IMG_SIZE, IMG_SIZE))
    for col_idx, m in enumerate(active_models):
        cam_np, pred = per_model_cams[m][i]
        overlay, _ = _cam_overlay(img_np, cam_np)
        axes[i, col_idx + 1].imshow(overlay)
        axes[i, col_idx + 1].set_title(f'{m}\nPred: {class_names[pred]}', fontsize=10)
        axes[i, col_idx + 1].axis('off')
        w = model_weights.get(m, 1/len(active_models))
        weighted_cam_sum += w * cam_np


    weighted_cam_sum = (weighted_cam_sum - weighted_cam_sum.min()) / (weighted_cam_sum.max() - weighted_cam_sum.min() + 1e-8)
    ens_overlay, _ = _cam_overlay(img_np, weighted_cam_sum)
    ens_pred = ensemble_pred_per_sample[i]
    axes[i, -1].imshow(ens_overlay)
    axes[i, -1].set_title(f'Ensemble\nPred: {class_names[ens_pred]}', fontsize=11, fontweight='bold')
    axes[i, -1].axis('off')

plt.suptitle('Ensemble Learning — Weighted Grad-CAM per Backbone + Fused',
             fontsize=18, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('ensemble_gradcam.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

print("\nSaved: ensemble_gradcam.png")

## 20. Package Output Files

In [ ]:
import zipfile

source_dir = '/kaggle/working/'
output_zip = '/kaggle/working/selected_files.zip'

extensions = ('.pth', '.png', '.csv')

with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(source_dir):
        if f.endswith(extensions):
            filepath = os.path.join(source_dir, f)
            zf.write(filepath, f)
            print(f"  Added: {f}")

print(f"\nZip created: {output_zip}")

from IPython.display import FileLink, display
display(FileLink('/kaggle/working/selected_files.zip'))

  Added: convnext_training_history.png
  Added: densenet121_confusion_matrix.png
  Added: fusion_mha_confusion_matrix.png
  Added: regnety008_training_history.png
  Added: best_densenet121_4class.pth
  Added: ensemble_confusion_matrices.png
  Added: densenet201_confusion_matrix.png
  Added: convnext_gradcam.png
